# Notebook 00 - Agent Registry v1 POC


In [1]:

# ============================================================
# EDIT THIS CELL ONLY - Agent Registry
# ============================================================
# POC note: secrets.yaml is plain text in Lakehouse Files.
# For production, switch generated secrets to Key Vault references.
#
# To add an agent:
#   1. Copy the Sparky dictionary.
#   2. Change agent_id, display_name, IDs, schema_name, and channel/auth values.
#   3. Add test cases for the new agent in test_cases.csv.
#   4. Run this notebook or run 00_orchestrator with RUN_CONFIG_REGISTRY=True.
# ============================================================

AGENT_REGISTRY = [
    {
        "agent_id": "sparky",
        "display_name": "Sparky",
        "enabled": True,
        "platform": "copilot_studio",
        "connection_mode": "direct_line_secret",
        "auth_mode": "direct_line_secret",
        "business_area": "Technical Support",
        "owner": "BI & AI Team",
        "schema_name": "cr578_Productsagent",
        "environment_id": "605e3ed6-b18f-ece1-ad54-4f71a003a6cb",
        "tenant_id": "226e353c-f71a-4b6a-a6af-293275183a60",
        "bot_id": "90834477-dcd9-4c4c-a025-dd256379a63a",
        "client_id": "90834477-dcd9-4c4c-a025-dd256379a63a",
        "direct_line_secret": "7OjuL5Med3ZDXtgkYFEwbReZ0Lgd6AeegEyyXngPmSvVBmCSdrVgJQQJ99CDACYeBjFAArohAAABAZBS3bsh.7KTe1rZPwCrfkKt12QHTRdYiyvIHasJ3tMW3xIQYVAXAPNvv36idJQQJ99CDACYeBjFAArohAAABAZBS34rL",
        "direct_connect_url": "https://605e3ed6b18fece1ad544f71a003a6.cb.environment.api.powerplatform.com/copilotstudio/dataverse-backed/authenticated/bots/cr578_Productsagent/conversations?api-version=2022-03-01-preview",
        "data_contracts": [],
        "deterministic_rules": [
            "no_internal_pricing_terms",
            "must_not_make_guaranteed_claims",
        ],
        "ragas_thresholds": {
            "faithfulness": 0.70,
            "answer_relevancy": 0.70,
        },
        "ms_eval_graders": [
            "General quality",
            "Keyword match",
        ],
        "microsoft_eval": {
            "test_set_ids": [],
            "pull_active_test_sets": True,
            "include_active_test_sets_only": True,
            "poc_mode": False,
            "mcs_connection_id": "",
            "required_methods": [
                "General quality",
                "Keyword match",
            ],
        },
        "risk_level": "standard",
        "p0_must_pass": True,
        "service_principal": {
            "client_id": "PASTE_SERVICE_PRINCIPAL_CLIENT_ID_HERE",
            "client_secret": "PASTE_SERVICE_PRINCIPAL_CLIENT_SECRET_HERE",
        },
    },
]

ALERT_WEBHOOK_URL = "PASTE_OPTIONAL_TEAMS_OR_POWER_AUTOMATE_WEBHOOK_URL_HERE"


StatementMeta(, 21b666e9-c98e-4c0a-bbfa-f369714adeb1, 3, Finished, Available, Finished, False)

In [2]:

# ============================================================
# Generated implementation cell - do not edit for normal agent changes
# ============================================================

try:
    environment
except NameError:
    environment = "dev"

import re
from pathlib import Path

import yaml

assert spark is not None, "Spark session not available - run this in Fabric."
from notebookutils import mssparkutils

LAKEHOUSE_NAME = "jacks_lakehouse"
ONELAKE_WORKSPACE_ID = "d9e51304-2b8a-4e62-b689-922e79fd76b4"
ONELAKE_LAKEHOUSE_ID = "537823c4-b83a-4a9b-9444-6039d55a4b9e"
LAKEHOUSE_DEFAULT_FILES_ROOT = "/lakehouse/default/Files"
LAKEHOUSE_FILES_ABFSS_ROOT = f"abfss://{ONELAKE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{ONELAKE_LAKEHOUSE_ID}/Files"
LAKEHOUSE_FILES_ROOT = LAKEHOUSE_FILES_ABFSS_ROOT
BASE_FILES_PATH = f"{LAKEHOUSE_FILES_ROOT}/agent_eval"
BASE_FILES_ABFSS_PATH = f"{LAKEHOUSE_FILES_ABFSS_ROOT}/agent_eval"
CONFIG_PATH = f"{BASE_FILES_PATH}/config"
RULES_PATH = f"{BASE_FILES_PATH}/rules"
SHARED_PATH = f"{BASE_FILES_PATH}/shared"
LOGS_PATH = f"{BASE_FILES_PATH}/logs"

AGENTS_YAML_PATH = f"{CONFIG_PATH}/agents.yaml"
AUTH_PROFILES_YAML_PATH = f"{CONFIG_PATH}/auth_profiles.yaml"
SECRETS_YAML_PATH = f"{CONFIG_PATH}/secrets.yaml"
MICROSOFT_EVAL_TEST_SETS_YAML_PATH = f"{CONFIG_PATH}/microsoft_eval_test_sets.yaml"
ALERT_CONFIG_YAML_PATH = f"{CONFIG_PATH}/alert_config.yaml"

AGENT_ID_PATTERN = r"^[a-z][a-z0-9_]*$"
GUID_PATTERN = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"


def ensure_dir(path):
    mssparkutils.fs.mkdirs(path)


def write_yaml(path, data):
    content = yaml.safe_dump(data, sort_keys=False, allow_unicode=False)
    mssparkutils.fs.put(path, content, True)


def file_exists(path):
    return mssparkutils.fs.exists(path)


def read_yaml(path, default):
    if not file_exists(path):
        return default
    return yaml.safe_load(mssparkutils.fs.head(path, 20 * 1024 * 1024)) or default


def is_placeholder(value):
    return not value or str(value).startswith("PASTE_")


def choose_secret_value(existing_secrets, key_name, proposed_value):
    if not is_placeholder(proposed_value):
        return proposed_value
    existing_value = existing_secrets.get(key_name)
    if existing_value and not is_placeholder(existing_value):
        return existing_value
    return proposed_value


def require(agent, field):
    value = agent.get(field)
    if value in (None, ""):
        raise ValueError(f"{agent.get('agent_id', '<unknown>')} missing required field: {field}")
    return value


def validate_guid(agent, field):
    value = require(agent, field)
    if not re.fullmatch(GUID_PATTERN, value):
        raise ValueError(f"{agent['agent_id']} field {field} must be GUID-shaped: {value}")


def validate_agent_registry(agents):
    seen = set()
    for agent in agents:
        agent_id = require(agent, "agent_id")
        if not re.fullmatch(AGENT_ID_PATTERN, agent_id):
            raise ValueError(f"Invalid agent_id {agent_id}; use lowercase letters, numbers, and underscores")
        if agent_id in seen:
            raise ValueError(f"Duplicate agent_id: {agent_id}")
        seen.add(agent_id)
        for field in ["display_name", "platform", "connection_mode", "schema_name"]:
            require(agent, field)
        for field in ["environment_id", "tenant_id", "bot_id", "client_id"]:
            validate_guid(agent, field)
        mode = agent.get("connection_mode")
        if mode == "direct_line_secret":
            require(agent, "direct_line_secret")
        elif mode == "copilot_direct_to_engine":
            require(agent, "direct_connect_url")
        else:
            raise ValueError(f"{agent_id} unsupported connection_mode for registry: {mode}")
        if agent.get("direct_connect_url") and not agent["direct_connect_url"].startswith("https://"):
            raise ValueError(f"{agent_id} direct_connect_url must start with https://")
    return True


def key(agent_id, suffix):
    return f"{agent_id}-{suffix}"


def agent_to_runtime_config(agent):
    agent_id = agent["agent_id"]
    ms_eval = agent.get("microsoft_eval") or {}
    connection_mode = agent["connection_mode"]
    auth_profile = f"{agent_id}_delegated_power_platform"
    auth_mode = "delegated_user_device_code"
    if connection_mode == "direct_line_secret":
        auth_profile = ""
        auth_mode = "direct_line_secret"

    return {
        "agent_id": agent_id,
        "display_name": agent["display_name"],
        "enabled": bool(agent.get("enabled", True)),
        "platform": agent["platform"],
        "connection_mode": connection_mode,
        "business_area": agent.get("business_area", ""),
        "owner": agent.get("owner", ""),
        "schema_name": agent["schema_name"],
        "auth_profile": auth_profile,
        "auth_mode": auth_mode,
        "direct_line_secret_key": key(agent_id, "direct-line-secret"),
        "direct_connect_url_key": key(agent_id, "direct-connect-url"),
        "copilot_test_set_id": agent.get("copilot_test_set_id", f"{agent_id.upper()}-POC-MS-SET-001"),
        "data_contracts": agent.get("data_contracts", []),
        "deterministic_rules": agent.get("deterministic_rules", []),
        "ragas_thresholds": agent.get("ragas_thresholds", {"faithfulness": 0.70, "answer_relevancy": 0.70}),
        "ms_eval_graders": agent.get("ms_eval_graders", ["General quality"]),
        "microsoft_eval": {
            "environment_id": agent["environment_id"],
            "bot_id": agent["bot_id"],
            "test_set_ids": ms_eval.get("test_set_ids", []),
            "pull_active_test_sets": bool(ms_eval.get("pull_active_test_sets", True)),
            "include_active_test_sets_only": bool(ms_eval.get("include_active_test_sets_only", True)),
            "poc_mode": bool(ms_eval.get("poc_mode", False)),
            "mcs_connection_id": ms_eval.get("mcs_connection_id", ""),
        },
        "risk_level": agent.get("risk_level", "standard"),
        "p0_must_pass": bool(agent.get("p0_must_pass", True)),
    }


def build_auth_profiles(agents):
    profiles = []
    for agent in agents:
        agent_id = agent["agent_id"]
        profiles.append({
            "name": f"{agent_id}_delegated_power_platform",
            "auth_mode": "delegated_user_device_code",
            "scope": "https://api.powerplatform.com/.default",
            "tenant_id": agent["tenant_id"],
            "client_id_key": key(agent_id, "client-id"),
        })
        profiles.append({
            "name": f"{agent_id}_service_principal_power_platform",
            "auth_mode": "service_principal_client_secret",
            "scope": "https://api.powerplatform.com/.default",
            "tenant_id_key": key(agent_id, "sp-tenant-id"),
            "client_id_key": key(agent_id, "sp-client-id"),
            "client_secret_key": key(agent_id, "sp-client-secret"),
        })
    return profiles


def build_secrets(agents, existing_secrets=None):
    existing_secrets = existing_secrets or {}
    secrets = {}
    for agent in agents:
        agent_id = agent["agent_id"]
        sp = agent.get("service_principal") or {}
        direct_line_key = key(agent_id, "direct-line-secret")
        secrets[direct_line_key] = choose_secret_value(existing_secrets, direct_line_key, agent.get("direct_line_secret", "PASTE_WEB_CHANNEL_SECRET_HERE"))
        secrets[key(agent_id, "direct-connect-url")] = agent.get("direct_connect_url", "")
        secrets[key(agent_id, "client-id")] = agent["client_id"]
        secrets[key(agent_id, "sp-tenant-id")] = agent["tenant_id"]
        sp_client_id_key = key(agent_id, "sp-client-id")
        sp_client_secret_key = key(agent_id, "sp-client-secret")
        secrets[sp_client_id_key] = choose_secret_value(existing_secrets, sp_client_id_key, sp.get("client_id", "PASTE_SERVICE_PRINCIPAL_CLIENT_ID_HERE"))
        secrets[sp_client_secret_key] = choose_secret_value(existing_secrets, sp_client_secret_key, sp.get("client_secret", "PASTE_SERVICE_PRINCIPAL_CLIENT_SECRET_HERE"))
    secrets["teams-webhook-url"] = choose_secret_value(existing_secrets, "teams-webhook-url", ALERT_WEBHOOK_URL)
    return secrets


def build_ms_eval_config(agents):
    return {
        "api_version": "2024-10-01",
        "pull_active_test_sets": True,
        "include_active_test_sets_only": True,
        "consolidate_to_unified_results": True,
        "imported_test_origin": "microsoft_eval",
        "poc_mode": False,
        "agents": {
            agent["agent_id"]: {
                "environment_id": agent["environment_id"],
                "bot_id": agent["bot_id"],
                "test_set_ids": (agent.get("microsoft_eval") or {}).get("test_set_ids", []),
                "mcs_connection_id": (agent.get("microsoft_eval") or {}).get("mcs_connection_id", ""),
                "required_methods": (agent.get("microsoft_eval") or {}).get("required_methods", ["General quality"]),
            }
            for agent in agents
        },
    }


def build_alert_config():
    return {
        "delta_table": {"enabled": True, "table": "agent_eval_alerts"},
        "webhook": {
            "enabled": False,
            "url_key": "teams-webhook-url",
            "max_rows": 20,
            "notes": "Set enabled=true after wiring this URL to Power Automate, Teams workflow, or another HTTP consumer.",
        },
    }


print("POC note: secrets.yaml is plain text in Lakehouse Files. For production, switch generated secrets to Key Vault references.")
for path in [BASE_FILES_PATH, CONFIG_PATH, RULES_PATH, SHARED_PATH, LOGS_PATH]:
    ensure_dir(path)

validate_agent_registry(AGENT_REGISTRY)
enabled_agents = [a for a in AGENT_REGISTRY if bool(a.get("enabled", True))]
existing_secret_values = (read_yaml(SECRETS_YAML_PATH, {"secrets": {}}).get("secrets") or {})

write_yaml(AGENTS_YAML_PATH, {"agents": [agent_to_runtime_config(a) for a in AGENT_REGISTRY]})
write_yaml(AUTH_PROFILES_YAML_PATH, {"auth_profiles": build_auth_profiles(AGENT_REGISTRY)})
write_yaml(SECRETS_YAML_PATH, {"secrets": build_secrets(AGENT_REGISTRY, existing_secret_values)})
write_yaml(MICROSOFT_EVAL_TEST_SETS_YAML_PATH, {"microsoft_eval": build_ms_eval_config(AGENT_REGISTRY)})
write_yaml(ALERT_CONFIG_YAML_PATH, {"alerts": build_alert_config()})

print(f"Wrote {len(AGENT_REGISTRY)} agent(s) to {AGENTS_YAML_PATH}")
print(f"Enabled agents: {[a['agent_id'] for a in enabled_agents]}")
print(f"Wrote auth profiles to {AUTH_PROFILES_YAML_PATH}")
print(f"Wrote plain-text POC secrets to {SECRETS_YAML_PATH}")
print(f"Wrote Microsoft Evaluation registry to {MICROSOFT_EVAL_TEST_SETS_YAML_PATH}")
print(f"Wrote alert config to {ALERT_CONFIG_YAML_PATH}")

try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit("PASS")
except ImportError:
    print("Standalone mode - would exit PASS")


StatementMeta(, 21b666e9-c98e-4c0a-bbfa-f369714adeb1, 4, Finished, Available, Finished, False)

POC note: secrets.yaml is plain text in Lakehouse Files. For production, switch generated secrets to Key Vault references.
Wrote 1 agent(s) to abfss://d9e51304-2b8a-4e62-b689-922e79fd76b4@onelake.dfs.fabric.microsoft.com/537823c4-b83a-4a9b-9444-6039d55a4b9e/Files/agent_eval/config/agents.yaml
Enabled agents: ['sparky']
Wrote auth profiles to abfss://d9e51304-2b8a-4e62-b689-922e79fd76b4@onelake.dfs.fabric.microsoft.com/537823c4-b83a-4a9b-9444-6039d55a4b9e/Files/agent_eval/config/auth_profiles.yaml
Wrote plain-text POC secrets to abfss://d9e51304-2b8a-4e62-b689-922e79fd76b4@onelake.dfs.fabric.microsoft.com/537823c4-b83a-4a9b-9444-6039d55a4b9e/Files/agent_eval/config/secrets.yaml
Wrote Microsoft Evaluation registry to abfss://d9e51304-2b8a-4e62-b689-922e79fd76b4@onelake.dfs.fabric.microsoft.com/537823c4-b83a-4a9b-9444-6039d55a4b9e/Files/agent_eval/config/microsoft_eval_test_sets.yaml
Wrote alert config to abfss://d9e51304-2b8a-4e62-b689-922e79fd76b4@onelake.dfs.fabric.microsoft.com/537823

In [3]:
from notebookutils import mssparkutils
import yaml

BASE = "abfss://d9e51304-2b8a-4e62-b689-922e79fd76b4@onelake.dfs.fabric.microsoft.com/537823c4-b83a-4a9b-9444-6039d55a4b9e/Files/agent_eval"

agents = yaml.safe_load(mssparkutils.fs.head(f"{BASE}/config/agents.yaml", 1024 * 1024))
secrets = yaml.safe_load(mssparkutils.fs.head(f"{BASE}/config/secrets.yaml", 1024 * 1024))

sparky = next(a for a in agents["agents"] if a["agent_id"] == "sparky")

print("connection_mode:", sparky.get("connection_mode"))
print("auth_mode:", sparky.get("auth_mode"))
print("direct_line_secret_key:", sparky.get("direct_line_secret_key"))
print("secret exists:", "sparky-direct-line-secret" in secrets.get("secrets", {}))


StatementMeta(, 21b666e9-c98e-4c0a-bbfa-f369714adeb1, 5, Finished, Available, Finished, False)

connection_mode: direct_line_secret
auth_mode: direct_line_secret
direct_line_secret_key: sparky-direct-line-secret
secret exists: True
